In [ ]:
from trellis2.models import SparseStructureFlowModel, SLatFlowModel, SparseStructureDecoder, FlexiDualGridVaeDecoder
from trellis2.modules.sparse import SparseTensor
from trellis2.modules.image_feature_extractor import DinoV3FeatureExtractor
from PIL import Image

from trellis2.pipelines.samplers import FlowEulerSampler, FlowEulerGuidanceIntervalSampler

import pyvista as pv
pv.set_jupyter_backend("html")

from safetensors.torch import load_file
import json
from pathlib import Path
import torch

In [ ]:
model_ckpts = {
    "sparse_structure_decoder": "ss_dec_conv3d_16l8_fp16",
    "sparse_structure_flow_model": "ss_flow_img_dit_1_3B_64_bf16",
    "shape_slat_decoder": "shape_dec_next_dc_f16c32_fp16",
    "shape_slat_flow_model_512": "slat_flow_img2shape_dit_1_3B_512_bf16",
    "shape_slat_flow_model_1024": "slat_flow_img2shape_dit_1_3B_1024_bf16",
}

model_cls = {
    "sparse_structure_decoder": SparseStructureDecoder,
    "sparse_structure_flow_model": SparseStructureFlowModel,
    "shape_slat_decoder": FlexiDualGridVaeDecoder,
    "shape_slat_flow_model_512": SLatFlowModel,
    "shape_slat_flow_model_1024": SLatFlowModel,
}


ckpt_path = Path("/flux/vault/pretrained_checkpoints/trellis")

def load_model(model_name: str):
    with open(ckpt_path / (model_ckpts[model_name] + ".json")) as f:
        model_args = json.load(f)
    model = model_cls[model_name](**model_args['args'])
    state_dict = load_file(ckpt_path / (model_ckpts[model_name] + ".safetensors"))
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    return model


In [ ]:
ss_flow_model = load_model("sparse_structure_flow_model")
ss_decoder = load_model("sparse_structure_decoder")
slat_flow_model = load_model("shape_slat_flow_model_512")
slat_decoder = load_model("shape_slat_decoder")

In [ ]:
NUM_SAMPLES = 1
SAMPLE_STEPS = 12

CONDITIONING_DIM = 1024
CONDITIONAL = True

cond = torch.zeros((NUM_SAMPLES, 1, CONDITIONING_DIM)).cuda()
if CONDITIONAL:
    # Optionally include image features
    image = Image.open("assets/example_image/T.png").convert("RGB")
    # will need to request access to the dinov3 HF repo
    dinov3 = DinoV3FeatureExtractor("facebook/dinov3-vitl16-pretrain-lvd1689m")
    dinov3.cuda()
    cond = dinov3([image]).repeat(NUM_SAMPLES, 1, 1)
    dinov3.cpu()

sampler = FlowEulerGuidanceIntervalSampler(sigma_min=1e-5)

In [ ]:
# Sample sparse structure latents
ss_flow_model.cuda()

reso = ss_flow_model.resolution
in_channels = ss_flow_model.in_channels
noise = torch.randn(NUM_SAMPLES, in_channels, reso, reso, reso).cuda()

ss_samples = sampler.sample(
    ss_flow_model,
    noise,
    cond=cond,
    neg_cond=torch.zeros_like(cond),
    steps=SAMPLE_STEPS,
    guidance_strength=7.5,
    guidance_rescale=0.7,
    guidance_interval=[0.6, 1.0],
    rescale_t=5.0,
).samples

ss_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
# Decode sparse structure latents
ss_decoder.cuda()

decoded = ss_decoder(ss_samples) > 0
if slat_flow_model.resolution != decoded.shape[2]:
    ratio = decoded.shape[2] // slat_flow_model.resolution
    decoded = torch.nn.functional.max_pool3d(decoded.float(), ratio, ratio, 0) > 0.5
coords = torch.argwhere(decoded)[:, [0, 2, 3, 4]].int()
ss_decoder.cpu()
torch.cuda.empty_cache()
print(coords.shape)

In [ ]:
# Sample shape slatents

slat_flow_model.cuda()

noise = SparseTensor(
            feats=torch.randn(coords.shape[0], slat_flow_model.in_channels).cuda(),
            coords=coords,
        )

slats = sampler.sample(
    model=slat_flow_model,
    noise=noise,
    cond=cond,
    neg_cond=torch.zeros_like(cond),
    steps=SAMPLE_STEPS,
    guidance_strength=7.5,
    guidance_rescale=0.7,
    guidance_interval=[0.6, 1.0],
    rescale_t=5.0,
).samples

slat_flow_model.cpu()
torch.cuda.empty_cache()

In [ ]:
def normalize_slats(slats):
    mean = [
            0.781296, 0.018091, -0.495192, -0.558457, 1.060530, 0.093252, 1.518149, -0.933218,
            -0.732996, 2.604095, -0.118341, -2.143904, 0.495076, -2.179512, -2.130751, -0.996944,
            0.261421, -2.217463, 1.260067, -0.150213, 3.790713, 1.481266, -1.046058, -1.523667,
            -0.059621, 2.220780, 1.621212, 0.877230, 0.567247, -3.175944, -3.186688, 1.578665
        ]
    std = [
            5.972266, 4.706852, 5.445010, 5.209927, 5.320220, 4.547237, 5.020802, 5.444004,
            5.226681, 5.683095, 4.831436, 5.286469, 5.652043, 5.367606, 5.525084, 4.730578,
            4.805265, 5.124013, 5.530808, 5.619001, 5.103930, 5.417670, 5.269677, 5.547194,
            5.634698, 5.235274, 6.110351, 5.511298, 6.237273, 4.879207, 5.347008, 5.405691
        ]
    
    std = torch.tensor(std)[None].cuda()
    mean = torch.tensor(mean)[None].cuda()
    
    return slats * std + mean

slats = normalize_slats(slats)

In [ ]:
# Decode shape slat
slat_decoder.cuda()
decoded_slats = slat_decoder(slats, return_subs=False)
slat_decoder.cpu()
torch.cuda.empty_cache()

In [ ]:
from typing import *
import numpy as np
import torch
import pyvista as pv
import cumesh


def postprocess_mesh(
    vertices: torch.Tensor,
    faces: torch.Tensor,
    aabb: Union[list, tuple, np.ndarray, torch.Tensor],
    voxel_size: Union[float, list, tuple, np.ndarray, torch.Tensor] = None,
    grid_size: Union[int, list, tuple, np.ndarray, torch.Tensor] = None,
    decimation_target: int = 1000000,
    remesh: bool = True,
    remesh_band: float = 1,
    remesh_project: float = 0,
    verbose: bool = True,
) -> pv.PolyData:
    """
    Clean, simplify, and optionally remesh an extracted mesh.
    Returns a PyVista PolyData triangle mesh.
    """
    device = vertices.device

    # --- Input Normalization ---
    if isinstance(aabb, (list, tuple)):
        aabb = np.array(aabb)
    if isinstance(aabb, np.ndarray):
        aabb = torch.tensor(aabb, dtype=torch.float32, device=device)

    if voxel_size is not None:
        if isinstance(voxel_size, float):
            voxel_size = [voxel_size] * 3
        if isinstance(voxel_size, (list, tuple)):
            voxel_size = np.array(voxel_size)
        if isinstance(voxel_size, np.ndarray):
            voxel_size = torch.tensor(voxel_size, dtype=torch.float32, device=device)
        grid_size = ((aabb[1] - aabb[0]) / voxel_size).round().int()
    else:
        assert grid_size is not None, "Either voxel_size or grid_size must be provided"
        if isinstance(grid_size, int):
            grid_size = [grid_size] * 3
        if isinstance(grid_size, (list, tuple)):
            grid_size = np.array(grid_size)
        if isinstance(grid_size, np.ndarray):
            grid_size = torch.tensor(grid_size, dtype=torch.int32, device=device)

    if verbose:
        print(f"Original mesh: {vertices.shape[0]} vertices, {faces.shape[0]} faces")

    vertices = vertices.cuda()
    faces = faces.cuda()

    mesh = cumesh.CuMesh()
    mesh.init(vertices, faces)

    # --- Initial Cleaning ---
    mesh.fill_holes(max_hole_perimeter=3e-2)
    if verbose:
        print(f"After filling holes: {mesh.num_vertices} vertices, {mesh.num_faces} faces")
    vertices, faces = mesh.read()

    # Build BVH (needed for remesh projection)
    if verbose:
        print("Building BVH...", end='', flush=True)
    bvh = cumesh.cuBVH(vertices, faces)
    if verbose:
        print("Done")

    # --- Branch 1: Standard Pipeline ---
    if not remesh:
        mesh.simplify(decimation_target * 3, verbose=verbose)
        if verbose:
            print(f"After initial simplification: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

        mesh.remove_duplicate_faces()
        mesh.repair_non_manifold_edges()
        mesh.remove_small_connected_components(1e-5)
        mesh.fill_holes(max_hole_perimeter=3e-2)
        if verbose:
            print(f"After initial cleanup: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

        mesh.simplify(decimation_target, verbose=verbose)
        if verbose:
            print(f"After final simplification: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

        mesh.remove_duplicate_faces()
        mesh.repair_non_manifold_edges()
        mesh.remove_small_connected_components(1e-5)
        mesh.fill_holes(max_hole_perimeter=3e-2)
        if verbose:
            print(f"After final cleanup: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

        mesh.unify_face_orientations()

    # --- Branch 2: Remeshing Pipeline ---
    else:
        center = aabb.mean(dim=0)
        scale = (aabb[1] - aabb[0]).max().item()
        resolution = grid_size.max().item()

        mesh.init(*cumesh.remeshing.remesh_narrow_band_dc(
            vertices, faces,
            center=center,
            scale=(resolution + 3 * remesh_band) / resolution * scale,
            resolution=resolution,
            band=remesh_band,
            project_back=remesh_project,
            verbose=verbose,
            bvh=bvh,
        ))
        if verbose:
            print(f"After remeshing: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

        mesh.simplify(decimation_target, verbose=verbose)
        if verbose:
            print(f"After simplifying: {mesh.num_vertices} vertices, {mesh.num_faces} faces")

    # --- Read final mesh & convert to PyVista ---
    out_vertices, out_faces = mesh.read()
    mesh.compute_vertex_normals()
    out_normals = mesh.read_vertex_normals()

    verts_np = out_vertices.cpu().numpy()
    faces_np = out_faces.cpu().numpy()
    normals_np = out_normals.cpu().numpy()

    # Coordinate system conversion (Y↔Z swap, invert Y) for GLB-style convention
    # Remove or adjust this block if you don't need the axis swap.
    verts_np[:, 1], verts_np[:, 2] = verts_np[:, 2].copy(), -verts_np[:, 1].copy()
    normals_np[:, 1], normals_np[:, 2] = normals_np[:, 2].copy(), -normals_np[:, 1].copy()

    # Build PyVista faces array: [3, v0, v1, v2, 3, v0, v1, v2, ...]
    n_faces = faces_np.shape[0]
    pv_faces = np.column_stack([np.full(n_faces, 3, dtype=np.int64), faces_np]).ravel()

    pv_mesh = pv.PolyData(verts_np, pv_faces)
    pv_mesh.point_data["Normals"] = normals_np

    if verbose:
        print(f"Final mesh: {pv_mesh.n_points} vertices, {pv_mesh.n_cells} faces")

    return pv_mesh

In [ ]:
decoded_slats

In [ ]:
n_cols = int(round(NUM_SAMPLES**0.5))
n_rows = (NUM_SAMPLES + n_cols - 1) // n_cols

plotter = pv.Plotter(shape=(n_rows, n_cols))

for i, mesh in enumerate(decoded_slats):
    mesh.fill_holes()
    mesh = postprocess_mesh(
        vertices=mesh.vertices,
        faces=mesh.faces,
        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        grid_size=slat_flow_model.resolution,
    )
    row = i // n_cols
    col = i % n_cols
    plotter.subplot(row, col)
    plotter.add_mesh(mesh, color='lightgrey')

plotter.link_views()
plotter.show()